# 07 — First blinded data round (2018A)

This is the first look at **real data** in the ABCD background-estimation program:
a deliberately small, safety-first slice — a deterministic **10% of 2018A**
(`event_number % 10 == 0`), with the signal-region box masked at processor level so
the SR yield is never formed. Its job is narrow and specific: **prove the blinding
machinery is correct on real data** before committing grid resources to the full
Run2018 campaign, and **scout** what the data can and cannot tell us at this size.

The upstream method study (notebooks 01–06) chose the plane, the SR anchor, and the
cosmic-veto variables on MC. Here we carry those exact definitions to data via the
same `study_setup` module, so nothing can drift.

**What this round establishes (and what it cannot).** With 10% of one era we can
verify the blinding is intact and characterize the sideband occupancy — but, as the
numbers below show, we cannot yet run the ABCD closure (the sidebands are starved)
nor validate the cosmic veto (the cosmic-enriched corner is inside the blind box).
Both point to concrete next data configurations, laid out at the end.

In [ ]:
import os, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import coffea.util

sys.path.insert(0, os.getcwd())
import abcd_tools as at
import study_setup as ss

hep.style.use("CMS")
plt.rcParams["figure.figsize"] = (9, 7)

# Blinded 2018A data: 10% decile (event_number % 10 == 0), unweighted, with the
# signal-region box masked at processor level (see PHASE3_SAFETY.md). MC is never
# masked; this notebook loads DATA only.
DATA_EOS = "root://cmseos.fnal.gov//store/group/lpcmetx/SIDM/coffea_outputs/murtazas/abcd_data_2018A"
SAMPLES = ["DoubleMuon_2018A_0", "DoubleMuon_2018A_1", "DoubleMuon_2018A_2"]
CH = {"2mu2e": "2mu2e_abcd_scan_data", "4mu": "4mu_abcd_scan_data"}
WD = "/uscms_data/d3/murtazas/abcd_data_local"

def load_data():
    os.makedirs(WD, exist_ok=True)
    tot = None
    for s in SAMPLES:
        loc = os.path.join(WD, s + ".coffea")
        if not os.path.exists(loc):
            subprocess.run(["xrdcp", "-s", f"{DATA_EOS}/{s}.coffea", loc], check=True)
        h = coffea.util.load(loc)[s]["hists"]
        tot = {n: v.copy() for n, v in h.items()} if tot is None else {n: tot[n] + v for n, v in h.items()}
    return tot

def bar_cat(ax, vals, edges, color, mark=None):
    """Bar plot over BIN INDEX (equal width) with edge-value tick labels -- the honest
    way to show these strongly non-uniform binnings (a continuous axis squishes them)."""
    n = len(vals)
    ax.bar(np.arange(n), vals, width=1.0, align="edge", color=color, alpha=0.65,
           edgecolor="k", lw=0.4)
    lab = ["%g" % e if abs(e) < 1000 else "1e%d" % int(round(np.log10(abs(e)))) for e in edges]
    ax.set_xticks(np.arange(n + 1)); ax.set_xticklabels(lab, rotation=45, fontsize=8, ha="right")
    ax.set_yscale("log"); ax.set_ylim(0.5, max(vals.max() * 2, 2))
    ax.margins(x=0.01)
    if mark is not None:
        mi = int(np.argmin(np.abs(np.asarray(edges) - mark)))
        ax.axvline(mi, color="k", ls="--", lw=1.4)

D = load_data()
print("blinded 2018A loaded:",
      "2mu2e =", int(D["abcd_scan_2mu2e_iso_iso"].sum().value), "events,",
      "4mu =", int(D["abcd_scan_4mu_iso_iso"].sum().value), "events")

## 1. Blinding tripwire

The blind box is defined on the fill quantities as **muiso < 0.5 AND mjj ≥ 50** —
region A at the loosest ladder rung, a *superset* of the SR at every working point
that will ever be examined. The processor drops those events before any histogram is
filled. The assertion below re-checks it directly on the merged data: summed over
every other axis, that quadrant must be exactly empty. This runs at load time in
every data notebook; if it ever fires, the round is aborted.

In [ ]:
# Blinding tripwire (PHASE3_SAFETY rule 4): the SR box is muiso < 0.5 AND mjj >= 50
# on the fill quantities, integrated over every event-cut axis. In data it MUST be
# empty. If this ever fires, stop and do not look further.
def sr_box_yield(hname, ch, xax):
    h = at.get_channel(D[hname], CH[ch])
    v, _, xe, ye = at.project_plane(h, xax, "mjj", {})   # integrate all other axes
    ix, iy = at.edge_index(xe, 0.5), at.edge_index(ye, 50.0)
    return float(v[:ix, iy:].sum())

for ch, hn, xa in [("2mu2e", "abcd_scan_2mu2e_iso_iso", "muiso"),
                   ("4mu",  "abcd_scan_4mu_iso_iso",  "muiso0")]:
    box = sr_box_yield(hn, ch, xa)
    assert box == 0, f"BLINDING LEAK in {ch}: SR box = {box}"
    print(f"{ch:6s}  SR-box yield = {box:.0f}   OK (blinded)")
print("\nBlinding intact on real data.")

## 2. Sideband occupancy — why the closure needs the full dataset

The ABCD prediction is `A_pred = B·C/D`, evaluated at the loosest healthy rung
(t=2.0: muiso < 0.5, mjj ≥ 50) **with the full SR event selection** applied —
tight EGM isolation, back-to-back Δφ, and displaced-muon requirements. Those cuts
are what make the region signal-like, and they are severe: on 10% of a single era
they leave the low-mass / high-isolation corners (C and D) empty.

**How to read the panels below.** Each is the 2×2 ABCD plane for one channel. A (top
left) is the SR — blinded, shown gray. B, C, D carry the visible data counts. With
D = 0 the prediction is undefined; this is a statistics limitation, not a method
failure. A meaningful data closure needs the **full Run2018** (all eras, all events)
to populate C and D to the registered health floor (n_eff ≥ 10 per region).

In [ ]:
# ABCD region yields at the loosest healthy rung (t=2.0): muiso<0.5, mjj>=50, WITH
# the SR event cuts (egm-iso<0.10, dphi>=2.0, displaced muons). A is the SR box ->
# blinded to 0; B, C, D are the visible sidebands; A_pred = B*C/D.
def data_regions(ch, plane):
    spec = ss.PLANES[ch][plane]
    scuts = ss.stage_menu(ch, plane)[-1][1]            # tightest event-cut menu (SR anchor)
    h = at.get_channel(D[spec["hist"]], CH[ch])
    sel = dict(scuts)
    for ax in spec["cuts"]:
        sel.setdefault(ax, "sum")
    v, var, xe, ye = at.project_plane(h, spec["x"], spec["y"], sel)
    xc = ss.snap_edge(xe[1:-1], spec["xspec"][1] * 2.0, tie="high")
    yc = ss.snap_edge(ye[1:-1], spec["yspec"][1] / 2.0, tie="low")
    return at.region_sums(v, var, xe, ye, (spec["xspec"][0], xc), (spec["yspec"][0], yc))

fig, axs = plt.subplots(1, 2, figsize=(15, 6.2))
for ax, (ch, plane, title) in zip(axs, [("2mu2e", "P4_muiso_mjj", "2mu2e  (muiso x mJJ)"),
                                        ("4mu", "Q6_iso0_mjj", "4mu  (lead muiso x mJJ)")]):
    reg = data_regions(ch, plane)
    pred, vpred = at.abcd_prediction(reg)
    cells = {"A": (0.5, 0.5, "#bbbbbb"), "B": (1.5, 0.5, "#5790fc"),
             "C": (0.5, 1.5, "#5790fc"), "D": (1.5, 1.5, "#f89c20")}
    for name, (x, y, col) in cells.items():
        val = reg[name][0]
        lab = "A = SR\n(BLINDED)" if name == "A" else f"{name} = {val:.0f}"
        ax.add_patch(plt.Rectangle((x - 0.5, y - 0.5), 1, 1, fc=col, ec="k", alpha=0.55))
        ax.text(x, y, lab, ha="center", va="center", fontsize=15, fontweight="bold")
    ax.axhline(1.0, color="k", lw=1); ax.axvline(1.0, color="k", lw=1)
    ax.set_xlim(0, 2); ax.set_ylim(0, 2)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(["muiso < 0.5", "muiso >= 0.5"])
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(["mjj >= 50", "mjj < 50"])
    pstr = "undefined (D=0)" if not np.isfinite(pred) else f"{pred:.2f}"
    # channel + prediction in the x-label so it never collides with the CMS label
    ax.set_xlabel(f"{title}\nA_pred = B*C/D = {pstr}", fontsize=12, labelpad=8)
hep.cms.label("Preliminary", data=True, com=13, ax=axs[0])
plt.tight_layout(); plt.show()
print("The sidebands are starved: C and D are empty at 10% of one era, so no data")
print("closure is possible here. Populating them requires the full Run2018 dataset.")

## 3. Cosmic variables in the sidebands — why the veto needs a dedicated CR

Cosmic-ray muons are the one background MC does not model: a single cosmic traversing
the detector is reconstructed as two nearly **back-to-back** muons
(cos α → −1), and — crucially — its upper and lower legs have very different impact
parameters, so the muons are **spatially incoherent** (large dz and vxy spread). A
genuine signal lepton-jet is the opposite: prompt and coherent. The Phase-2 veto
therefore combines a back-to-back tag (cos α, computed against *extra* muons so real
back-to-back 4mu signal does not self-veto) with the internal spatial spread.

**But there is a structural problem, visible in the numbers below.** A cosmic is
back-to-back *and* isolated, which puts it at **low isolation and high di-object
mass** — precisely the SR box. So cosmics are preferentially **blinded out** of these
sidebands. The distributions confirm it: only a handful of events are back-to-back,
and essentially none of those are displaced — the cosmic-enriched population is not
here to be studied.

**How to read the figure.** Top row 2mu2e, bottom row 4mu; columns are cos α, dz
spread, vxy spread. Each variable spans a huge dynamic range on strongly
non-uniform bins, so it is drawn over **bin index** (equal-width bars) with the bin
edge values labeled on the axis; the y scale is log. The dashed line marks the cos α
veto candidate (−0.98, left column) and an illustrative spatial-incoherence scale
(1 cm, spread columns). Note the large-spread tail in dz/vxy that is *not*
back-to-back — an uninterpretable mix (DSA dz resolution, mis-clustering) that only a
cosmic-dominated control region can resolve.

The consequence: validating the cosmic veto in data requires a **dedicated cosmic
CR** that *selects* the cosmic topology instead of blinding it (the standard
back-to-back-muon control region), run without the SR box because it is
background-only and signal-free by construction.

In [ ]:
# Cosmic variables in the visible sidebands. A cosmic ray is two back-to-back muons
# (cos(alpha) -> -1) that are spatially INCOHERENT (large dz / vxy spread between the
# LJ muons). Signal LJ muons are prompt and coherent. These distributions test
# whether a cosmic population even survives into the blinded sidebands.
def axis1d(hname, ch, axname):
    h = at.get_channel(D[hname], CH[ch])
    v, _, xe, _ = at.project_plane(h, axname, "parity", {})
    return v.sum(axis=1), xe

specs = [("abcd_cosmic_2mu2e", "2mu2e", "mincosa", "dzspread", "vxyspread"),
         ("abcd_cosmic_4mu",  "4mu",  "mincosa0", "dzspread0", "vxyspread0")]
fig, axs = plt.subplots(2, 3, figsize=(16, 9))
labels = ["min cos(alpha)  (LJ muon vs extra muon)", "LJ muon dz spread [cm]",
          "LJ muon vxy spread [cm]"]
marks = [-0.98, 1.0, 1.0]           # cos-a veto candidate; spread "incoherent" scale (illustrative)
for r, (hn, ch, cosa, dzs, vxs) in enumerate(specs):
    for c, (axn, lab, mk) in enumerate(zip((cosa, dzs, vxs), labels, marks)):
        ax = axs[r, c]
        v, xe = axis1d(hn, ch, axn)
        bar_cat(ax, v, xe, "#5790fc" if c == 0 else "#e42536", mark=mk)
        ax.set_xlabel(lab, fontsize=11)
        if c == 0:
            ax.set_ylabel(f"{ch} events")
hep.cms.label("Preliminary", data=True, com=13, ax=axs[0, 0])
plt.tight_layout(); plt.show()

# quantify the cosmic-tagged tail
for hn, ch, cosa, dzs, vxs in specs:
    h = at.get_channel(D[hn], CH[ch])
    vv, _, xe, ye = at.project_plane(h, cosa, dzs, {})
    i98 = at.edge_index(xe, -0.98); jdz = at.edge_index(ye, 0.1)
    tot, bb = vv.sum(), vv[:i98, :].sum()
    print(f"{ch:6s}: back-to-back (cos<-0.98) = {bb:.0f}/{tot:.0f}; "
          f"of those, dz_spread>0.1cm = {vv[:i98, jdz:].sum():.0f}")

## 4. What this round established, and the next data configurations

**Established (on real data):** the blinding is correct — the SR box is exactly
empty, the 10% decile and the data-safe pipeline work end to end, and 979 (2mu2e) /
299 (4mu) events flow to the visible sidebands. The infrastructure is proven; the
expensive campaign can be launched on it with confidence.

**Cannot be done at this size, with the reason and the fix:**

| Goal | Blocked by | Next data configuration |
|---|---|---|
| Data ABCD closure | sidebands starved (C = D = 0) | **Full Run2018** — same blinded config, all eras/events, to populate B/C/D to n_eff ≥ 10 |
| Cosmic-veto validation | cosmics live in the (blinded) SR box | **Dedicated cosmic CR** — a back-to-back-muon selection, un-blinded because signal-free, to measure the veto's efficiency and rejection |
| Data validation-region closure | the inverted-Δφ VR's A-corner is also blinded | **Δφ-conditional mask** — blind only the back-to-back (Δφ ≥ 2.0) signal-like corner (after checking signal leakage below the cut in MC) |

Each is a re-run and each touches blinding, so each is designed and signed off
before it runs. The SR itself stays blinded throughout; nothing in this program
forms an SR data yield until the collaboration approves unblinding.